In [49]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import seaborn as sns
from  sklearn import  linear_model
from sklearn import neighbors
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder

titanic=pd.read_csv('Titanic-Dataset.csv')


In [50]:
titanic

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [51]:
#הורדת עמודה שרב ערכיה היו חסרים
titanic
#t = titanic.drop(columns=['Cabin'])
t=titanic
t


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [52]:
#בדיקה איפה חסרים ערכים
t.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [53]:
#השלמת ערכים חסרים לפי ממוצע
age_mean=SimpleImputer(strategy='mean')
t['Age']=age_mean.fit_transform(t[['Age']])

#השלמת ערכים לפי שכיח
Embarked_mode=SimpleImputer(strategy='most_frequent')
t['Embarked']=Embarked_mode.fit_transform(t[['Embarked']]).ravel()

t.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

In [54]:
#קידוד
t = pd.get_dummies(t, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True, dtype=int)

X= t.drop(['Survived','Name','Ticket','PassengerId', 'Cabin'],axis=1)
#נרמול
scaler_minmax = MinMaxScaler().set_output(transform="pandas")
X = scaler_minmax.fit_transform(X)

In [55]:
 #בדיקת בנירמול-לבדוק שהערכים בין 0 ל1
print(X.agg(['min', 'max']))


     Age  SibSp  Parch  Fare  Sex_male  Embarked_Q  Embarked_S  Pclass_2  \
min  0.0    0.0    0.0   0.0       0.0         0.0         0.0       0.0   
max  1.0    1.0    1.0   1.0       1.0         1.0         1.0       1.0   

     Pclass_3  
min       0.0  
max       1.0  


In [56]:
print(t.head())

   PassengerId  Survived                                               Name  \
0            1         0                            Braund, Mr. Owen Harris   
1            2         1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3         1                             Heikkinen, Miss. Laina   
3            4         1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4            5         0                           Allen, Mr. William Henry   

    Age  SibSp  Parch            Ticket     Fare Cabin  Sex_male  Embarked_Q  \
0  22.0      1      0         A/5 21171   7.2500   NaN         1           0   
1  38.0      1      0          PC 17599  71.2833   C85         0           0   
2  26.0      0      0  STON/O2. 3101282   7.9250   NaN         0           0   
3  35.0      1      0            113803  53.1000  C123         0           0   
4  35.0      0      0            373450   8.0500   NaN         1           0   

   Embarked_S  Pclass_2  Pclass_3  
0       

In [57]:
print(t.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Name         891 non-null    object 
 3   Age          891 non-null    float64
 4   SibSp        891 non-null    int64  
 5   Parch        891 non-null    int64  
 6   Ticket       891 non-null    object 
 7   Fare         891 non-null    float64
 8   Cabin        204 non-null    object 
 9   Sex_male     891 non-null    int64  
 10  Embarked_Q   891 non-null    int64  
 11  Embarked_S   891 non-null    int64  
 12  Pclass_2     891 non-null    int64  
 13  Pclass_3     891 non-null    int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 97.6+ KB
None


In [58]:
#X_test X_train y_test y_train
y=t['Survived']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [59]:
#knn-בודק לפי הכמות שכנים שלו מי הכי דומה לו ומסיק אם ניצל או לא
model=KNeighborsClassifier(n_neighbors=5)
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

print(f"---KNN Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("\n")

---KNN Metrics ---
Accuracy:  0.804
Precision: 0.783
Recall:    0.730
F1-Score:  0.755




In [60]:
#Decision Tree עץ החלטה-בונה עץ של שאלות הוא מהיר אבל נוטה להיות קיצוני ולטעות
model=DecisionTreeClassifier()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print(f"--- Decision Tree Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("\n")

--- Decision Tree Metrics ---
Accuracy:  0.760
Precision: 0.707
Recall:    0.716
F1-Score:  0.711




In [61]:
#SVM- -כלומר מי שרד ומי לא מודל שיוצר גבול הכי ברור ורחוק בין הקבוצות
model=SVC(kernel='linear')
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print(f"---SVM Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("\n")

---SVM Metrics ---
Accuracy:  0.782
Precision: 0.754
Recall:    0.703
F1-Score:  0.727




In [62]:
#RandomForest-יוצר מספר עצים וכל עץ עם שינוי קטן בנתונים ובסוף הוא בודק  את התשובה הרווחת בין העצים

param_grid = {
    'n_estimators': [50, 100, 200],      # מספר העצים ביער
    'max_depth': [None, 10, 20, 30],     # עומק מקסימלי של כל עץ
    'min_samples_split': [2, 5, 10],     # מינימום דגימות לפיצול צומת
    'criterion': ['gini', 'entropy']     # מדד איכות הפיצול
}
# 4. הרצת החיפוש
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

# 5. הצגת התוצאות
print(f"הפרמטרים הטובים ביותר שנמצאו: {grid_search.best_params_}")
print(f"דיוק (Accuracy) ב-Cross-Validation: {grid_search.best_score_:.2f}")

# בדיקת המודל הטוב ביותר על נתוני הטסט
best_rf = grid_search.best_estimator_
print(f"דיוק סופי על נתוני הטסט: {best_rf.score(X_test, y_test):.2f}")

print(f"---RandomForest Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("\n")

Fitting 5 folds for each of 72 candidates, totalling 360 fits
הפרמטרים הטובים ביותר שנמצאו: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
דיוק (Accuracy) ב-Cross-Validation: 0.83
דיוק סופי על נתוני הטסט: 0.83
---RandomForest Metrics ---
Accuracy:  0.782
Precision: 0.754
Recall:    0.703
F1-Score:  0.727


